# <center><span style="color:RED"><u><br/>Web Scraping des effectifs de la Premier League</u></span></center>

## <span style="color:RED">1-Introduction</span>

L'objectif de ce notebook est de scraper les effectifs de tous les clubs du championnat anglais, appelé Premier League, durant la saison 2025-2026.

Pour chaque joueur, nous souhaitons récupérer les informations suivantes : le club, le nom, le prénom, l'âge, la nationalité et la valeur marchande. À noter que le championnat compte 20 équipes, composées chacune d'environ 25 à 30 joueurs.

Ce notebook constitue la première étape de notre projet visant à expliquer la valeur marchande des joueurs de football. Les données seront récupérées sur Transfermarkt, nettoyées, puis complétées avec les performances des joueurs issues de FBref.

Sources utilisées pour construire ce notebook :
- https://pythonds.linogaliana.fr/content/manipulation/04a_webscraping_TP.html
- https://www.transfermarkt.com/premier-league/startseite/wettbewerb/GB1/saison_id/2025

## <span style="color:RED">2-Importation des bibliothèques</span>

En amont, il faut installer les quatre bibliothèques externes ci-dessous afin de pouvoir les importer. Pour ce faire, il faut lancer dans le terminal la commande suivante :

```bash
pip install pandas requests beautifulsoup4 lxml
```

**Remarques :**

- La bibliothèque `time` est déjà intégrée à Python. Elle sera utilisée pour espacer les requêtes de plusieurs secondes afin d'éviter de surcharger le serveur.
- La bibliothèque `requests` permettra d'envoyer des requêtes HTTP au serveur et de récupérer le contenu HTML de la page web.
- La bibliothèque `bs4` (BeautifulSoup) permettra de parcourir le document HTML et de rechercher des éléments à l'aide des balises et de leurs attributs.
- La bibliothèque `lxml` servira de parseur HTML pour BeautifulSoup, afin d'analyser et de structurer le contenu récupéré.
- La bibliothèque `pandas` servira à créer des DataFrames et à effectuer des opérations sur les tableaux de données.

In [2]:
import pandas as pd
import requests
import bs4
import time

## <span style="color:RED">3-Récupération des clubs de Premier League avec leurs URL</span>

Dans un premier temps, nous allons partir de la page qui recense tous les clubs de Premier League (présentée en introduction) afin de construire un dictionnaire ayant pour clés les noms des clubs et pour valeurs les URL menant à leurs pages Transfermarkt.

Ces URL seront ensuite utilisées pour accéder aux effectifs de chaque club.

In [5]:
url_premier_league="https://www.transfermarkt.com/premier-league/startseite/wettbewerb/GB1/saison_id/2025"

request_text=requests.get(url_premier_league,headers={"User-Agent": "Mozilla/5.0"}).content

page=bs4.BeautifulSoup(request_text,"lxml")

tableau_clubs=page.find("table",{"class": "items"})

rows=tableau_clubs.find("tbody").find_all("tr")
#print(rows[1])

FeatureNotFound: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?

In [ ]:
clubs={}
for row in rows:
    club=row.find("td",{"class":"hauptlink"})
    if club is not None:
        club_name=club.text.strip()
        club_link=club.find("a").get("href")
        clubs[club_name]="https://www.transfermarkt.com"+club_link

print("Nombres de clubs récupérés :", len(clubs))
clubs

Nombres de clubs récupérés : 20


{'Manchester City': 'https://www.transfermarkt.com/manchester-city/startseite/verein/281/saison_id/2025',
 'Arsenal FC': 'https://www.transfermarkt.com/fc-arsenal/startseite/verein/11/saison_id/2025',
 'Chelsea FC': 'https://www.transfermarkt.com/fc-chelsea/startseite/verein/631/saison_id/2025',
 'Liverpool FC': 'https://www.transfermarkt.com/fc-liverpool/startseite/verein/31/saison_id/2025',
 'Tottenham Hotspur': 'https://www.transfermarkt.com/tottenham-hotspur/startseite/verein/148/saison_id/2025',
 'Manchester United': 'https://www.transfermarkt.com/manchester-united/startseite/verein/985/saison_id/2025',
 'Crystal Palace': 'https://www.transfermarkt.com/crystal-palace/startseite/verein/873/saison_id/2025',
 'Newcastle United': 'https://www.transfermarkt.com/newcastle-united/startseite/verein/762/saison_id/2025',
 'AFC Bournemouth': 'https://www.transfermarkt.com/afc-bournemouth/startseite/verein/989/saison_id/2025',
 'Nottingham Forest': 'https://www.transfermarkt.com/nottingham-fo

## <span style="color:RED">4-Création d'une fonction de scraping des effectifs</span>

Afin d'automatiser la récupération des informations des joueurs des 20 clubs, il est judicieux de développer une fonction qui pourra être utilisée dans une boucle par la suite.

C'est l'objectif de cette section : nous allons construire une fonction scrape_team() qui permettra de récupérer, pour un club et une URL donnés (passés en arguments), les informations de l'ensemble de ses joueurs en 2025.

In [ ]:
def scrape_team(club,url)->pd.DataFrame:
    """
    Récupère les informations de tous les joueurs d'un club disponible sur Transfermarkt.

    Args:
        club (str): Nom du club à scrapper.
        url (str): URL Transfermarkt du club à scrapper.

    Returns:
        pd.DataFrame: DataFrame contenant les informations de tous les joueurs du club disponible sur Transfermarkt.
    """
    url=url.replace("/startseite/","/kader/").split("?")[0]+"?saison_id=2025"
    request_text=requests.get(url,headers={"User-Agent": "Mozilla/5.0"}).content
    page=bs4.BeautifulSoup(request_text,"lxml")
    tableau_joueurs=page.find("table",{"class":"items"})
    rows=tableau_joueurs.find("tbody").find_all("tr")

    players = []
    for row in rows:
        cells=row.find_all("td")
        player_name=row.find("td",{"class":"hauptlink"})
        nationality_flag=row.find("img",{"class":"flaggenrahmen"})
        #On conserve uniquement les cellules ayant les informations sur les joueurs, c'est à dire contenant au moins 9 cellules.
        if player_name is not None and len(cells)>=9:
             players.append({
                "club":club,
                "joueur":player_name.text.strip(),
                "age":cells[5].text.strip(),
                "nationalite":nationality_flag.get("title") if nationality_flag else None,
                "valeur_marchande":cells[-1].text.strip()
            })
    return pd.DataFrame(players)

## <span style="color:RED">5-Construction du Dataset de Premier League</span>

Pour conclure, nous allons appliquer notre fonction aux 20 clubs présents en Premier League à l'aide d'une boucle. Cela permettra de créer 20 DataFrames (un pour chaque club), que nous allons ensuite concaténer afin d'obtenir un unique DataFrame regroupant l'ensemble des joueurs. Enfin, nous enregistrerons ce dernier sous la forme d'un fichier CSV.

In [ ]:
all_teams=[]
for club,url in clubs.items():
    print("Scraping : ",club)
    df_team=scrape_team(club,url)
    all_teams.append(df_team)
    time.sleep(2)

df_premier_league=pd.concat(all_teams,ignore_index=True)

df_premier_league.head()

Scraping :  Manchester City
Scraping :  Arsenal FC
Scraping :  Chelsea FC
Scraping :  Liverpool FC
Scraping :  Tottenham Hotspur
Scraping :  Manchester United
Scraping :  Crystal Palace
Scraping :  Newcastle United
Scraping :  AFC Bournemouth
Scraping :  Nottingham Forest
Scraping :  Brighton & Hove Albion
Scraping :  Aston Villa
Scraping :  Brentford FC
Scraping :  West Ham United
Scraping :  Everton FC
Scraping :  Sunderland AFC
Scraping :  Wolverhampton Wanderers
Scraping :  Fulham FC
Scraping :  Leeds United
Scraping :  Burnley FC


,club,joueur,age,nationalite,valeur_marchande
0,Manchester City,Gianluigi Donnarumma,27,Italy,€45.00m
1,Manchester City,James Trafford,23,England,€25.00m
2,Manchester City,Ederson,32,Brazil,€10.00m
3,Manchester City,Stefan Ortega,33,Germany,€4.00m
4,Manchester City,Marcus Bettinelli,34,England,€300k


In [ ]:
df_premier_league.describe()

,club,joueur,age,nationalite,valeur_marchande
count,798,798,798,798,798
unique,20,781,26,74,67
top,Liverpool FC,Stefan Ortega,19,England,-
freq,45,2,68,301,98


À l'aide des statistiques descriptives ci-dessus, nous pouvons établir quelques premiers constats :
- Nous avons bien 20 clubs.
- Sur les 798 joueurs recensés, 781 sont uniques. Il faudra donc étudier l'origine des doublons.
- Pour la variable `valeur_marchande`, il semble y avoir des valeurs manquantes qu'il faudra imputer par la suite.

## <span style="color:RED">6-Nettoyage lèger du Dataset de Premier League</span>

Avant d'enregistrer le dataset obtenu, nous allons effectuer un premier léger nettoyage des données afin de faciliter le croisement avec nos autres données. Le but de ce premier nettoyage est de régler les soucis qui peuvent se faire immédiatement sans besoin d'étude au préalable, comme par exemple convertir des variables qui sont censées être des variables numériques en variables numériques, regarder quels sont les doublons, les valeurs manquantes et déterminer la variable qui va nous permettre de faire une jointure avec nos autres données, etc.

## <span style="color:RED">6.1-Conversion des variables dans le bon format</span>

In [ ]:
df_premier_league.info()

<class 'pandas.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   club              798 non-null    str  
 1   joueur            798 non-null    str  
 2   age               798 non-null    str  
 3   nationalite       798 non-null    str  
 4   valeur_marchande  798 non-null    str  
dtypes: str(5)
memory usage: 31.3 KB


À l'aide de la commande ci-dessus, on peut remarquer qu'actuellement, on a 5 colonnes qui sont toutes interprétées comme des chaînes de caractères. Certes, les variables `club`, `joueur` et `nationalite` sont des chaînes de caractères. Cependant, `age` et la `valeur_marchande` doivent etre des variables numériques, il faut donc les convertir.

In [ ]:
#Convertion de l'age en numérique
df_premier_league['age']=pd.to_numeric(df_premier_league['age'],errors='coerce')
#Conversion de la valeur marchande en numérique
df_premier_league['valeur_marchande']=(
    df_premier_league['valeur_marchande']
    .str.replace('€','',regex=False)
    .str.replace('m','e6',regex=False)
    .str.replace('k','e3',regex=False)
    .replace('-',None)
    .pipe(pd.to_numeric,errors='coerce')
    .astype('Int64')
)

## <span style="color:RED">6.2-Repérage des valeurs manquantes et doublons</span>

In [ ]:
df_premier_league.info()

<class 'pandas.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   club              798 non-null    str  
 1   joueur            798 non-null    str  
 2   age               798 non-null    int64
 3   nationalite       798 non-null    str  
 4   valeur_marchande  700 non-null    Int64
dtypes: Int64(1), int64(1), str(3)
memory usage: 32.1 KB


Après avoir converti nos colonnes aux formats appropriés, nous constatons que la variable `valeur_marchande` contient 98 valeurs manquantes. Nous déciderons de la méthode à adopter pour traiter ces valeurs une fois la fusion des jeux de données effectuée.

À présent nous allons voir quels sont nos doublons et essayer de comprendre la cause.

In [ ]:
df_premier_league.nunique() #17 Joueurs doublons

club                 20
joueur              781
age                  26
nationalite          74
valeur_marchande     66
dtype: int64

In [ ]:
doublons=df_premier_league[
    df_premier_league.duplicated(subset=["joueur"], keep=False)
]

doublons.sort_values("joueur")

,club,joueur,age,nationalite,valeur_marchande
718,Fulham FC,Adama Traoré,30,Spain,6000000
567,West Ham United,Adama Traoré,30,Spain,6000000
359,AFC Bournemouth,Antoine Semenyo,26,Ghana,80000000
38,Manchester City,Antoine Semenyo,26,Ghana,80000000
361,AFC Bournemouth,Ben Gannon-Doak,20,Scotland,15000000
161,Liverpool FC,Ben Gannon-Doak,20,Scotland,15000000
278,Crystal Palace,Brennan Johnson,25,Wales,30000000
208,Tottenham Hotspur,Brennan Johnson,25,Wales,30000000
388,Nottingham Forest,Douglas Luiz,28,Brazil,18000000
472,Aston Villa,Douglas Luiz,28,Brazil,18000000


Nous avons donc identifié 17 joueurs apparaissant en doublon dans notre jeu de données. Après vérification, il ne s’agit pas d’erreurs de saisie. En effet, ces doublons s’expliquent par le fait que ces joueurs ont changé de club au cours de la saison 2025-2026, passant d’un club de Premier League à un autre.

Par exemple, Adama Traoré (le premier doublon identifié dans notre tableau) a commencé la saison au Fulham FC avant de rejoindre West Ham United le 28 janvier 2026. Il apparaît donc deux fois dans notre jeu de données, une fois pour chacun des clubs dans lesquels il a évolué. Cette situation est également valable pour les 16 autres joueurs concernés.

Pour le moment, nous décidons de conserver ces doublons. Nous les comparerons ensuite avec les données issues de FBref afin de définir une règle de décision cohérente permettant de déterminer quelle observation conserver pour chaque joueur (celle correspondant à son premier ou à son deuxième club). L’objectif final est d’obtenir un jeu de données dans lequel chaque joueur est représenté par une seule ligne, tout en garantissant la cohérence des informations après la fusion des différentes sources de données.

## <span style="color:RED">6.3-Création de la variable de fusion</span>

In [ ]:
df_premier_league.to_csv(
    "premier_league_2025_2026.csv",
    index=False)